# Estimator Model Test Notebook

This notebook is only for the three model passes used by the estimator pipeline:

1. Vision model: image -> visual description
2. Text model: description -> clean keywords
3. Text model: keywords + context -> final estimate

How to run:

- Install the Python deps from `tools/requirements-price-estimator.txt`.
- Make sure Ollama is reachable at your `OLLAMA_HOST` value.
- Set `image_path` in the model cells before running them.

In [12]:
import os

os.environ.setdefault('OLLAMA_HOST', 'http://192.168.0.102:11434')
os.environ.setdefault('VISION_MODEL', 'llava-phi3')
os.environ.setdefault('TEXT_MODEL', 'llama3.2')

print('OLLAMA_HOST =', os.environ['OLLAMA_HOST'])
print('VISION_MODEL =', os.environ['VISION_MODEL'])
print('TEXT_MODEL   =', os.environ['TEXT_MODEL'])

import ollama
import ddgs

client = ollama.Client(host=os.environ['OLLAMA_HOST'])
vision_model = os.environ['VISION_MODEL']
text_model = os.environ['TEXT_MODEL']

OLLAMA_HOST = http://192.168.0.102:11434
VISION_MODEL = llava-phi3
TEXT_MODEL   = llama3.2


In [24]:
image_path = "../photos/turtle.jpeg"
vision_res = client.chat(
    model=vision_model,
    messages=[
        {
            "role": "user",
            "content": (
                "Analyze this image for crochet pattern matching.\n\n"

                "Describe ONLY what is clearly visible.\n"
                "Do not guess.\n"
                "Do not explain your reasoning.\n"
                "Do not mention the background.\n\n"

                "Return exactly this format:\n\n"

                "Object: <main item>\n"
                "Craft: <amigurumi/plushie/doll/bag/etc>\n"
                "Subject: <animal/object/character>\n"
                "Features: <comma separated distinctive features>\n"
                "Colors: <comma separated colors>\n\n"

                "Good example:\n"
                "Object: turtle plush\n"
                "Craft: amigurumi\n"
                "Subject: turtle\n"
                "Features: round shell, short legs, large eyes\n"
                "Colors: green, blue"
            ),
            "images": [image_path],
        }
    ],
)

visual_description = vision_res["message"]["content"]
print(visual_description)

Object: stuffed animal
Craft: amigurumi
Subject: sea turtle
Features: round body, pointy nose, big black eyes
Colors: blue, gray


In [6]:
cleaner_res = client.chat(
    model=text_model,
    messages=[
        {
            "role": "user",
            "content": (
                "Extract crochet pattern search terms from the description.\n\n"

                "Rules:\n"
                "- Output ONLY the keywords.\n"
                "- No explanations.\n"
                "- No introductory text.\n"
                "- No bullet points.\n"
                "- No labels.\n"
                "- No sentences.\n"
                "- No markdown.\n"
                "- No confidence statements.\n"
                "- No notes about excluded attributes.\n"
                "- Return a single comma-separated line.\n\n"

                "Include only terms that someone would realistically type when searching for a crochet pattern.\n"
                "Prioritize item type, subject, style, shape, and distinctive features.\n"
                "Ignore background, lighting, image quality, and non-searchable attributes.\n"
                "Avoid generic terms such as 'handcrafted', 'high complexity', 'yarn', 'crochet', or 'made by hand' unless they are essential.\n\n"

                "Example output:\n"
                "amigurumi turtle, sea turtle, crochet turtle plush, realistic turtle, turtle plushie\n\n"

                f"Description:\n{visual_description}"
            )
        }
    ],
)

keywords = cleaner_res["message"]["content"].strip()

In [14]:
print(keywords)

amigurumi seal, seal doll, round seal, spotted seal, plush seal


In [25]:
import json
import requests
import ollama

def search_web(query):
    url = "https://api.tavily.com/search"
    payload = {"api_key": "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", "query": query}
    response = requests.post(url, json=payload)
    return response.json().get("results", [])

def run_agent(user_prompt):
    messages = [{"role": "user", "content": user_prompt}]
    search_tool = {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web for real-time information and pricing details",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"],
            }
        }
    }
    response = ollama.chat(
        model="llama3.1",
        messages=messages,
        tools=[search_tool]
    )
    tool_calls = response.get("message", {}).get("tool_calls", [])
    if not tool_calls:
        return response["message"]["content"]
    messages.append(response["message"])
    for tool in tool_calls:
        if tool["function"]["name"] == "search_web":
            query = tool["function"]["arguments"]["query"]
            search_results = search_web(query)
            messages.append({
                "role": "tool",
                "content": json.dumps(search_results)
            })
    final_response = ollama.chat(model="llama3.1", messages=messages)
    return final_response["message"]["content"]


In [27]:
dimensions = '12 cm tall'

search_query = f'{visual_description} {dimensions} pricing specifications materials'
results = search_web(search_query)

In [28]:
print(results)

[{'url': 'https://sheepandsquirrel.com/products/turtle', 'title': 'Sea Turtle Amigurumi Crochet Stuffed Animal - Sheep & Squirrel, LLC', 'content': 'These adorable amigurumi crochet stuffed animal sea turtles make a wonderful gift or a cute addition to any stuffed animal collection.', 'score': 0.53050464, 'raw_content': None}, {'url': 'https://www.ebay.com/itm/325866311166', 'title': 'Crochet Amigurumi Plush Large Baby Blue And Sprinkles Sea Turtle ...', 'content': 'This handmade crochet amigurumi plush sea turtle is the perfect addition to any collection. With its baby blue and sprinkles design, it brings a touch of whimsy', 'score': 0.42176673, 'raw_content': None}, {'url': 'https://www.etsy.com/market/plush_sea_turtle', 'title': 'Plush Sea Turtle - Etsy', 'content': 'Discover charming plush sea turtles, perfect for nurseries, gifts, and ocean lovers. Explore handmade crochet creations and cuddly stuffed animals.', 'score': 0.39985907, 'raw_content': None}, {'url': 'https://www.youtu

In [42]:
import re
import os
import subprocess
from pprint import pprint

# Require transcript API to be installed; allow ImportError to surface
from youtube_transcript_api import YouTubeTranscriptApi


def extract_youtube_urls(results):
    # assume iterable; allow errors to surface
    results_list = list(results)
    urls = set()
    yt_re = re.compile(r"https?://(?:www\.)?(?:youtube\.com/watch\?v=|youtu\.be/)[^\s'\"]+")
    md_link_re = re.compile(r"\[([^\]]+)\]\(([^\)]+)\)")

    def add_candidate(value):
        if not isinstance(value, str):
            return
        for match in yt_re.findall(value):
            urls.add(match)
        for label, target in md_link_re.findall(value):
            for match in yt_re.findall(label):
                urls.add(match)
            for match in yt_re.findall(target):
                urls.add(match)

    fields = ['url', 'link', 'title', 'body', 'snippet', 'source']
    for r in results_list:
        if isinstance(r, dict):
            for k in fields:
                add_candidate(r.get(k))
            for v in r.values():
                add_candidate(v)
        elif isinstance(r, str):
            add_candidate(r)

    return sorted(urls)


def get_video_id(url):
    m = re.search(r"v=([A-Za-z0-9_-]{6,})", url)
    if m:
        return m.group(1)
    m = re.search(r"youtu\.be/([A-Za-z0-9_-]{6,})", url)
    if m:
        return m.group(1)
    return None


def fetch_transcript_for_url(url):
    vid = get_video_id(url)
    if not vid:
        raise ValueError(f'Cannot extract video id from {url}')
    ytt_api = YouTubeTranscriptApi()
    transcript = ytt_api.fetch(vid)
    text = "\n".join(t['text'] for t in transcript)
    return {'id': vid, 'transcript': transcript, 'text': text}


def download_youtube_video(url, out_file='downloaded_video.mp4'):
    url = url.strip()
    cmd = [
        'yt-dlp',
        '--no-playlist',
        '--force-ipv4',
        '--extractor-args', 'youtube:player_client=android,web',
        '-f', 'bestvideo*+bestaudio/best',
        '--merge-output-format', 'mp4',
        '-o', out_file,
        url,
    ]
    print('Downloading:', url)
    # Let subprocess errors surface
    subprocess.run(cmd, check=True)


# Use existing `results` if present, else run a fresh web search (requires `keywords` and `search_web`).
results_local = globals().get('results')
if results_local is None:
    if not globals().get('keywords'):
        raise RuntimeError('No `results` and no `keywords` found — run vision+cleaner cells first.')
    dimensions = globals().get('dimensions', '12 cm tall')
    query = f"{keywords} {dimensions} pricing specifications materials"
    print('Running `search_web` for query:', query)
    results_local = search_web(query)

# Extract YouTube URLs
yt_urls = extract_youtube_urls(results_local)
if not yt_urls:
    print('No YouTube URLs found in search results.')
else:
    print('Found YouTube URLs:')
    for i, u in enumerate(yt_urls, 1):
        print(f'{i}. {u}')

    # Prefer transcript-first: try each URL until we find a transcript
    transcript_data = None
    chosen_url = None
    for u in yt_urls:
        # allow any exceptions from transcript fetch to surface
        t = fetch_transcript_for_url(u)
        transcript_data = t
        chosen_url = u
        break

    if transcript_data:
        print('\nTranscript found for', chosen_url)
        print('Transcript snippet:')
        print(transcript_data['text'][:1000])
        globals()['video_transcript'] = transcript_data
    else:
        chosen = yt_urls[0]
        out_file = 'downloaded_video.mp4'
        download_youtube_video(chosen, out_file)
        if 'estimate_hours_from_video' in globals():
            print('\nEstimating hours from downloaded video...')
            res = estimate_hours_from_video(out_file, complexity_factor=1.1)
            pprint(res)
        else:
            print('Video estimator function not found. Ensure the video-analysis cell has been executed.')


Found YouTube URLs:
1. https://www.youtube.com/watch?v=pnSH57FLbTk


TypeError: 'FetchedTranscriptSnippet' object is not subscriptable

In [34]:
extract_youtube_urls(results)

['https://www.youtube.com/watch?v=pnSH57FLbTk']

In [37]:
from youtube_transcript_api import YouTubeTranscriptApi
import re
id = re.search(r"[?&]v=([^&]+)", extract_youtube_urls(results)[0]).group(1)

ytt_api = YouTubeTranscriptApi()
ytt_api.fetch(id)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hello everybody welcome and welcome back', start=0.08, duration=5.279), FetchedTranscriptSnippet(text='to my youtube creative studio', start=2.56, duration=4.96), FetchedTranscriptSnippet(text='this is a qc', start=5.359, duration=3.441), FetchedTranscriptSnippet(text='turtle', start=7.52, duration=4.079), FetchedTranscriptSnippet(text="and i love it and it's a great project", start=8.8, duration=5.6), FetchedTranscriptSnippet(text='for summer holiday what do you think you', start=11.599, duration=4.881), FetchedTranscriptSnippet(text='can use this cute', start=14.4, duration=5.84), FetchedTranscriptSnippet(text='crochet sea turtle as a gift or as well', start=16.48, duration=6.959), FetchedTranscriptSnippet(text='as a summer decor', start=20.24, duration=5.68), FetchedTranscriptSnippet(text='the plaster nose to this video tutorial', start=23.439, duration=4.561), FetchedTranscriptSnippet(text='is available in the information b

In [ ]:
vid = get_video_id("https://www.youtube.com/watch?v=pnSH57FLbTk")
if not vid:
    raise ValueError(f'Cannot extract video id from {url}')
ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(vid)
text = "\n".join(snippet.text for snippet in transcript.snippets)
print(text)

In [ ]:
# if your variable is `transcript`


hello everybody welcome and welcome back
to my youtube creative studio
this is a qc
turtle
and i love it and it's a great project
for summer holiday what do you think you
can use this cute
crochet sea turtle as a gift or as well
as a summer decor
the plaster nose to this video tutorial
is available in the information box
below this video and the written button
at my website please find the link also
below in the information box
you will
need three millimeters crochet hole if
you want you can use safety plastic eyes
eight millimeters or you can crochet
eyes with me please prepare some
scissors suppress three needles
markers and stuffing
you can use any yarn you like for
toys crocheting i use this
um
100 percent cotton yar in one bowl which
is 50 grams is
155
meters but you can use any yarn you long
leg
i will do the measurement at the end of
my video tutorial
we will start with a
shell and let's make a magic ring
let's start with chain three
which counts as a first double crochet
one tw